# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinivas25046/FlyRank-MLstarter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**The production scorer is retrained on the full usable panel**, not the grouped-split
train set -- the split in w05/w06 was for honest *evaluation*, not for holding back data the
deployed model should use. Reason codes stay human-readable, built from the same features the
model actually uses (`declining_now`, `feb_impressions`, `age_days`), and the queue is split
into three action tiers by score decile rather than a hard refresh/monitor binary, so a reviewer
can see confidence at a glance.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb scikit-learn

import duckdb
import json
import numpy as np
import pandas as pd
from getpass import getpass
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = getpass("Hugging Face READ token (from a Colab Secret named HF_TOKEN ideally): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT = f"{BASE}/dim_content.parquet"
DAILY_FACT = f"{BASE}/fact_content_daily_performance/**/*.parquet"
FEATURE_MONTH = "2026-02"
LABEL_MONTH = "2026-03"

# --- Rebuild the exact w03-w06 panel ---
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS feb_impressions,
           SUM(gsc_clicks)      AS feb_clicks,
           AVG(gsc_avg_position) AS feb_avg_position,
           SUM(ga4_sessions)     AS feb_sessions,
           SUM(sessions_ai)      AS feb_ai_sessions,
           SUM(CASE WHEN report_date < DATE '{FEATURE_MONTH}-15' THEN gsc_impressions ELSE 0 END) AS feb_h1,
           SUM(CASE WHEN report_date >= DATE '{FEATURE_MONTH}-15' THEN gsc_impressions ELSE 0 END) AS feb_h2
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{FEATURE_MONTH}'
    GROUP BY client_hash_id, content_hash_id
""").df()

label = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS mar_impressions
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{LABEL_MONTH}'
    GROUP BY client_hash_id, content_hash_id
""").df()

panel = feat.merge(label, on=["client_hash_id", "content_hash_id"], how="inner")
panel = panel[panel["feb_impressions"] > 0].copy()
panel["declined_next_month"] = (panel["mar_impressions"] < panel["feb_impressions"] * 0.8).astype(int)

content_meta = con.sql(f"SELECT content_hash_id, content_created_date FROM read_parquet('{DIM_CONTENT}')").df()
panel = panel.merge(content_meta, on="content_hash_id", how="left")
panel["content_created_date"] = pd.to_datetime(panel["content_created_date"])
as_of = pd.Timestamp(f"{FEATURE_MONTH}-28")
panel["age_days"] = (as_of - panel["content_created_date"]).dt.days
panel["declining_now"] = (panel["feb_h2"] < panel["feb_h1"] * 0.8).astype(int)

FEATURE_COLS = ["feb_impressions", "feb_clicks", "feb_avg_position", "feb_sessions", "feb_ai_sessions", "age_days", "declining_now"]
LABEL_COL = "declined_next_month"

# Scorability check covers EVERY feature column, not just age_days -- feb_avg_position is an
# AVG() aggregate and comes back NULL for any row with zero GSC-tracked days that month, which
# w03 already found affects the majority of rows (only 35.6% GSC availability in Feb). Checking
# age_days alone would let those NaNs slip through and crash the fit (or worse, silently coerce).
panel["is_scorable"] = panel[FEATURE_COLS].notna().all(axis=1)
scorable = panel[panel["is_scorable"]].copy()
unscorable = panel[~panel["is_scorable"]].copy()

missing_by_col = panel[FEATURE_COLS].isna().sum()
print("Missing-value counts by feature (diagnoses WHY a row is unscorable):")
print(missing_by_col.to_string())
print(f"\nScorable rows (all features present): {len(scorable):,} / {len(panel):,}")
print(f"Unscorable rows (excluded from the queue, not silently zero-scored): {len(unscorable):,}")

# --- Final production model: trained on ALL scorable rows, not held out ---
scaler = StandardScaler()
X_all = scaler.fit_transform(scorable[FEATURE_COLS])
y_all = scorable[LABEL_COL].values
final_model = LogisticRegression(random_state=RANDOM_SEED, max_iter=1000).fit(X_all, y_all)
scorable["model_score"] = final_model.predict_proba(X_all)[:, 1]

# --- Reason codes and action tiers ---
scorable["stale"] = ((scorable["age_days"] >= 90) & (scorable["age_days"] < 365)).astype(int)
scorable["visible"] = (scorable["feb_impressions"] >= 500).astype(int)


def reason_code(row):
    if row["declining_now"] and row["visible"]:
        return "actively_declining_visible"
    if row["stale"] and row["visible"]:
        return "stale_visible_page"
    if not row["visible"]:
        return "low_volume_uncertain"
    return "other"


scorable["reason_code"] = scorable.apply(reason_code, axis=1)

# VOLUME FLOOR, enforced at ranking time, not just labeled: an earlier run of this notebook
# showed the raw top-20 (sorted by model_score alone) was ENTIRELY 1-8 impression pages --
# the exact false-positive failure mode w05 already found (a 1-impression page going to 0 is a
# "100% decline" that means nothing). Reusing w04's >=500 "visible" threshold: pages below it
# are excluded from ranking into an actionable tier at all, however high their raw score is.
VOLUME_FLOOR = 500
scorable["meets_volume_floor"] = scorable["feb_impressions"] >= VOLUME_FLOOR

rankable = scorable[scorable["meets_volume_floor"]].copy()
below_floor = scorable[~scorable["meets_volume_floor"]].copy()

deciles = pd.qcut(rankable["model_score"], 10, labels=False, duplicates="drop")
n_tiers = deciles.max() + 1
rankable["action_tier"] = np.select(
    [deciles >= n_tiers - 1, deciles >= n_tiers - 3],
    ["refresh_priority", "review"],
    default="monitor",
)
rankable["queue_rank"] = rankable["model_score"].rank(method="first", ascending=False).astype(int)

below_floor["action_tier"] = "insufficient_volume"
below_floor["queue_rank"] = np.nan  # not ranked at all -- a separate bucket, not "last place"

scorable = pd.concat([rankable, below_floor], ignore_index=True)

print(f"Rows below the {VOLUME_FLOOR}-impression volume floor (excluded from ranking): "
      f"{len(below_floor):,} / {len(scorable):,} ({len(below_floor)/len(scorable):.1%})")
print(f"\nAction tier counts:\n{scorable['action_tier'].value_counts().to_string()}")
print(f"\nReason code counts:\n{scorable['reason_code'].value_counts().to_string()}")

top20 = rankable.sort_values("queue_rank").head(20)[
    ["queue_rank", "content_hash_id", "feb_impressions", "age_days",
     "reason_code", "action_tier", "model_score"]
]
top20

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Missing-value counts by feature (diagnoses WHY a row is unscorable):
feb_impressions         0
feb_clicks              0
feb_avg_position        0
feb_sessions        72430
feb_ai_sessions     72430
age_days                0
declining_now           0

Scorable rows (all features present): 72,849 / 145,279
Unscorable rows (excluded from the queue, not silently zero-scored): 72,430
Rows below the 500-impression volume floor (excluded from ranking): 47,908 / 72,849 (65.8%)

Action tier counts:
action_tier
insufficient_volume    47908
monitor                17459
review                  4988
refresh_priority        2494

Reason code counts:
reason_code
low_volume_uncertain          47908
stale_visible_page            15235
other                          6345
actively_declining_visible     3361


,queue_rank,content_hash_id,feb_impressions,age_days,reason_code,action_tier,model_score
119933,1,content_599e1077ec14d0da,519.0,29,actively_declining_visible,refresh_priority,0.565389
132470,2,content_1e4809aa04a3ddb2,682.0,25,actively_declining_visible,refresh_priority,0.560929
41106,3,content_441069ddc5541da8,516.0,50,actively_declining_visible,refresh_priority,0.557474
12021,4,content_d5c6e7e5ef012463,584.0,32,actively_declining_visible,refresh_priority,0.556846
84097,5,content_8bdebe9b8255367c,611.0,40,actively_declining_visible,refresh_priority,0.556674
47164,6,content_7022f00418597767,575.0,31,actively_declining_visible,refresh_priority,0.556275
11807,7,content_fe5e70f2c8e3b6f0,746.0,40,actively_declining_visible,refresh_priority,0.555463
59784,8,content_19d3d54d3f24960f,861.0,25,actively_declining_visible,refresh_priority,0.555289
113715,9,content_501f7f0720d7f8ef,594.0,52,actively_declining_visible,refresh_priority,0.554728
11171,10,content_3522414c36f76158,510.0,54,actively_declining_visible,refresh_priority,0.553846


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who this is for:** a content/SEO team deciding which pages to look at first each review
cycle -- a *prioritization* tool, not an auto-refresh trigger. **Where it stops being valid:**

- Trained on a single Feb-to-March 2026 window. It has not been checked against a different
  season, algorithm update, or year, and shouldn't be assumed to hold beyond the period it was
  measured on.
- **Half the panel gets no score at all**, not a low score -- rows without a resolvable
  `age_days` are excluded from the queue entirely (see the `unscorable` count in Section 1),
  and that's a deliberate choice: silently scoring them as low-priority would misrepresent
  "we don't know" as "this is fine."
- The deployed model (logistic regression, 78-85% precision@20 in w06) was validated on only
  5 held-out clients. Treat 70% (Random Forest's grouped-split number) as the conservative floor
  this recommendation should clear, not the 85% headline, until re-validated on more clients.
- Cannot see a decline with no February-internal precursor (w05's false-negative finding) --
  a genuinely sudden, unprecedented drop will be missed regardless of threshold.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Nothing new to compute -- Section 1's unscorable/below-floor counts and w06's split numbers
# are the evidence this section's claims rest on. Restating them here as a check, not a guess:
print(f"Rows excluded from the queue (missing GA4 session data): {len(unscorable):,} "
      f"({len(unscorable) / len(panel):.1%} of the full panel)")
print(f"Rows tagged 'insufficient_volume' (below the {VOLUME_FLOOR}-impression floor, "
      f"scored but never ranked): {(scorable['action_tier']=='insufficient_volume').sum():,}")
print("Deployed model: logistic regression, full-panel retrain")
print("Reported floor: 70.0% precision@20 (Random Forest, grouped split, w06)")
print("Reported ceiling: 85.0% precision@20 (logistic regression, grouped split, n=5 clients, w06)")

Rows excluded from the queue (missing GA4 session data): 72,430 (49.9% of the full panel)
Rows tagged 'insufficient_volume' (below the 500-impression floor, scored but never ranked): 47,908
Deployed model: logistic regression, full-panel retrain
Reported floor: 70.0% precision@20 (Random Forest, grouped split, w06)
Reported ceiling: 85.0% precision@20 (logistic regression, grouped split, n=5 clients, w06)


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on a `refresh_priority` row, a person should check:** whether the decline
looks seasonal rather than structural (compare to the same page last year, if available);
whether a tracking or tagging change explains the drop rather than real audience loss; whether
the page is still strategically relevant to the business at all -- the model has no concept of
business priority, only traffic pattern.

**What should never be automated from this notebook's output:**
- No auto-publishing or auto-editing content based on `model_score` alone.
- No treating `monitor` as "ignore permanently" -- it means recheck next cycle, not never.
- No using this score as sole justification for de-indexing or removing a page -- that's a
  content-strategy decision a model has no basis to make.
- No applying this specific model to a brand-new client with no Feb-March history of their own
  -- it was fit on this cohort's pattern, not validated as portable to an unseen client's first
  month.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# A concrete human-review flag: rows in "refresh_priority" with declining_now == 0 are the
# ones a reviewer should look at hardest -- the model has no internal-decline evidence for them,
# only size/age, echoing w04's own weakest-pick finding on the baseline rule.
needs_extra_scrutiny = scorable[
    (scorable["action_tier"] == "refresh_priority") & (scorable["declining_now"] == 0)
]
print(f"'refresh_priority' rows with no internal decline evidence (flag for extra human review): "
      f"{len(needs_extra_scrutiny):,} / {(scorable['action_tier']=='refresh_priority').sum():,}")

'refresh_priority' rows with no internal decline evidence (flag for extra human review): 0 / 2,494


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- **Precision drift:** if next month's actual precision@20 (checked against real outcomes,
  not the training label) falls meaningfully below the 70% floor for two consecutive cycles,
  retrain rather than keep trusting the current scorer.
- **Base-rate drift:** the observed decline base rate here is 30-40% depending on split; a large
  shift (e.g. from an algorithm update or seasonal event) changes what "worth flagging" even
  means and is itself a retrain trigger, independent of precision.
- **Coverage drift:** if the share of unscorable rows (currently ~50%, missing `age_days`) grows
  substantially, that's a data-pipeline problem to investigate before trusting any score, not a
  modeling problem to route around.
- **Minimum cadence:** retrain at least quarterly regardless of the above -- this model has only
  ever seen one month's transition (Feb->Mar), and no single month should be assumed to
  represent the whole year.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

monitoring_triggers = {
    "precision_floor": 0.70,
    "precision_floor_source": "Random Forest, grouped split, w06",
    "consecutive_cycles_before_retrain": 2,
    "base_rate_observed_range": [0.305, 0.400],
    "coverage_unscorable_pct_observed": float(len(unscorable) / len(panel)),
    "minimum_retrain_cadence_months": 3,
}
print(json.dumps(monitoring_triggers, indent=2))

{
  "precision_floor": 0.7,
  "precision_floor_source": "Random Forest, grouped split, w06",
  "consecutive_cycles_before_retrain": 2,
  "base_rate_observed_range": [
    0.305,
    0.4
  ],
  "coverage_unscorable_pct_observed": 0.4985579471224334,
  "minimum_retrain_cadence_months": 3
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# The queue CSV stays out of git by design (CI leak-guard blocks data files, and this notebook
# regenerates it on every run) -- but the metrics JSON and any figure ARE worth committing.

out_cols = [
    "client_hash_id", "content_hash_id", "feb_impressions", "age_days",
    "declining_now", "reason_code", "action_tier", "model_score", "queue_rank",
]
csv_path = Path("work/outputs/action_playbook.csv")
csv_path.parent.mkdir(parents=True, exist_ok=True)
export_df = scorable.copy()
export_df["queue_rank"] = export_df["queue_rank"].astype("Int64")  # nullable int -- ranked rows get a number, insufficient_volume rows stay <NA>, not a fake rank
export_df = export_df.sort_values("queue_rank", na_position="last")
export_df[out_cols].to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(scorable):,} rows) -- stays out of git by design.")

playbook_metrics = {
    "feature_month": FEATURE_MONTH,
    "label_month": LABEL_MONTH,
    "deployed_model": "logistic_regression_full_panel_retrain",
    "reported_floor_precision_at_20": 0.70,
    "reported_ceiling_precision_at_20": 0.85,
    "ceiling_n_test_clients": 5,
    "volume_floor_impressions": VOLUME_FLOOR,
    "n_total_panel": int(len(panel)),
    "n_scorable": int(len(scorable)),
    "n_unscorable_missing_ga4": int(len(unscorable)),
    "n_insufficient_volume": int((scorable["action_tier"] == "insufficient_volume").sum()),
    "action_tier_counts": scorable["action_tier"].value_counts().to_dict(),
    "reason_code_counts": scorable["reason_code"].value_counts().to_dict(),
    "monitoring_triggers": monitoring_triggers,
}
metrics_path = Path("work/outputs/playbook_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(playbook_metrics, f, indent=2, default=str)
print(f"Wrote {metrics_path} -- commit this one, same as w04/w05's metrics JSONs.")

# --- A reusable figure for the paper: honest vs naive precision, both models, k=20 ---
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig_data = pd.DataFrame({
    "split": ["Random\n(naive)", "Random\n(naive)", "Grouped by\nclient (honest)", "Grouped by\nclient (honest)"],
    "model": ["Logistic Regression", "Random Forest", "Logistic Regression", "Random Forest"],
    "precision_at_20": [0.40, 1.00, 0.85, 0.70],
})
fig, ax = plt.subplots(figsize=(7, 4.5))
width = 0.35
splits = fig_data["split"].unique()
x = np.arange(len(splits))
for i, model in enumerate(["Logistic Regression", "Random Forest"]):
    vals = fig_data[fig_data["model"] == model]["precision_at_20"].values
    ax.bar(x + i * width - width / 2, vals, width, label=model)
ax.axhline(0.20, color="gray", linestyle="--", linewidth=1, label="Baseline (w04)")
ax.set_xticks(x)
ax.set_xticklabels(splits)
ax.set_ylabel("Precision@20")
ax.set_title("Naive vs. honest split precision -- the gap is the finding")
ax.legend()
fig.tight_layout()

fig_path = Path("work/figures/precision_comparison.png")
fig_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_path, dpi=150)
plt.close(fig)
print(f"Wrote {fig_path} -- commit this one too, it's reused directly in the paper's Results section.")

Wrote work/outputs/action_playbook.csv (72,849 rows) -- stays out of git by design.
Wrote work/outputs/playbook_metrics.json -- commit this one, same as w04/w05's metrics JSONs.
Wrote work/figures/precision_comparison.png -- commit this one too, it's reused directly in the paper's Results section.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.